In [1]:
%pip install numpy pandas matplotlib scikit-learn tensorflow keras-tuner
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GRU,
    Dense,
    Dropout,
    LayerNormalization
)
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)
from tensorflow.keras.optimizers import Adam
import keras_tuner as kt

df = pd.read_csv("water_consumption_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"])
df.sort_values("Date", inplace=True)
df.reset_index(drop=True, inplace=True)

target = "Total_Daily_Usage_ML"

df['month_sin'] = np.sin(2*np.pi*df['Month']/12)
df['month_cos'] = np.cos(2*np.pi*df['Month']/12)
df['day_sin'] = np.sin(2*np.pi*df['Day_of_Year']/365)
df['day_cos'] = np.cos(2*np.pi*df['Day_of_Year']/365)

for lag in [1,2,3,4,5,7,14,21,30,60,90,180,365]:
    df[f'lag_{lag}'] = df[target].shift(lag)

df['ma_7'] = df[target].rolling(7).mean()
df['ma_30'] = df[target].rolling(30).mean()
df['std_7'] = df[target].rolling(7).std()
df['std_30'] = df[target].rolling(30).std()
df['ema_7'] = df[target].ewm(span=7).mean()
df['ema_30'] = df[target].ewm(span=30).mean()
df['diff_1'] = df[target].diff(1)
df['diff_7'] = df[target].diff(7)
df['pct_change_1'] = df[target].pct_change(1)
df['pct_change_7'] = df[target].pct_change(7)

df.dropna(inplace=True)

features = [
    'Day_of_Week', 'Month', 'Year', 'Day_of_Year', 'Is_Weekend', 'Holiday_Flag',
    'Temperature_C', 'Rainfall_mm', 'Humidity_pct', 'Evaporation_Rate_mm',
    'Population', 'Household_Count', 'Urbanization_Index', 'Pump_Status', 'Reservoir_Level_pct',
    'month_sin', 'month_cos', 'day_sin', 'day_cos',
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_7', 'lag_14', 'lag_21', 'lag_30', 'lag_60', 'lag_90', 'lag_180', 'lag_365',
    'ma_7', 'ma_30', 'std_7', 'std_30', 'ema_7', 'ema_30', 'diff_1', 'diff_7', 'pct_change_1', 'pct_change_7'
]

split_row = int(len(df) * 0.8)
train_df = df.iloc[:split_row]
test_df = df.iloc[split_row:]

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

feature_scaler.fit(train_df[features])
target_scaler.fit(train_df[[target]])

scaled_features = feature_scaler.transform(df[features])
scaled_target = target_scaler.transform(df[[target]])

window_size = 14

def create_sequences(features, target, window_size):
    X, y = [], []
    for i in range(window_size, len(features)):
        X.append(features[i-window_size:i])
        y.append(target[i])
    return np.array(X), np.array(y)

X_all, y_all = create_sequences(scaled_features, scaled_target, window_size)

sequence_split = split_row - window_size
X_train = X_all[:sequence_split]
y_train = y_all[:sequence_split]
X_test = X_all[sequence_split:]
y_test = y_all[sequence_split:]

validation_size = int(len(X_train) * 0.1)
X_fit = X_train[:-validation_size]
y_fit = y_train[:-validation_size]
X_val = X_train[-validation_size:]
y_val = y_train[-validation_size:]

def build_model(hp):
    model = Sequential()
    
   
    model.add(GRU(
        units=hp.Int('gru_1_units', min_value=32, max_value=128, step=32),
        return_sequences=True,
        input_shape=(X_train.shape[1], X_train.shape[2])
    ))
    if hp.Boolean('use_ln_1'):
        model.add(LayerNormalization())
    model.add(Dropout(rate=hp.Float('dropout_1', min_value=0.0, max_value=0.3, step=0.1)))

    model.add(GRU(
        units=hp.Int('gru_2_units', min_value=16, max_value=64, step=16),
        return_sequences=False
    ))
    if hp.Boolean('use_ln_2'):
        model.add(LayerNormalization())
    model.add(Dropout(rate=hp.Float('dropout_2', min_value=0.0, max_value=0.3, step=0.1)))
 
    model.add(Dense(
        units=hp.Int('dense_units', min_value=16, max_value=64, step=16), 
        activation='relu'
    ))
    

    model.add(Dense(1))
    

    lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 5e-4])
    
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='mse',
        metrics=['mae']
    )
    return model

tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=12,           
    executions_per_trial=1,
    directory='gru_tuning_dir',
    project_name='water_demand_gru'
)


tune_early_stop = EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
tune_reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

print("--- Initiating GRU Hyperparameter Exploration ---")
tuner.search(
    X_fit, y_fit,
    validation_data=(X_val, y_val),
    epochs=60,                
    batch_size=32,
    shuffle=False,           
    callbacks=[tune_early_stop, tune_reduce_lr],
    verbose=1
)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n--- Tuning Optimization Complete! Best Hyperparameters: ---")
print(f"-> GRU Layer 1 Units: {best_hps.get('gru_1_units')}")
print(f"-> Use LayerNorm 1  : {best_hps.get('use_ln_1')}")
print(f"-> GRU Layer 2 Units: {best_hps.get('gru_2_units')}")
print(f"-> Dense Layer Units: {best_hps.get('dense_units')}")
print(f"-> Learning Rate    : {best_hps.get('learning_rate')}")

print("\n--- Training Best Configured Model for Full Epochs ---")
model = tuner.hypermodel.build(best_hps)

final_early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)
final_reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1)

history = model.fit(
    X_fit, y_fit,
    validation_data=(X_val, y_val),
    epochs=300,
    batch_size=32,
    shuffle=False,
    callbacks=[final_early_stop, final_reduce_lr],
    verbose=1
)

predictions = model.predict(X_test)
predictions_actual = target_scaler.inverse_transform(predictions)
y_test_actual = target_scaler.inverse_transform(y_test)

mae = mean_absolute_error(y_test_actual, predictions_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, predictions_actual))
r2 = r2_score(y_test_actual, predictions_actual)

print("\nRESULTS (OPTIMIZED GRU MODEL)")
print("=============================")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")
print("=============================")



Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Reloading Tuner from gru_tuning_dir\water_demand_gru\tuner0.json
--- Initiating GRU Hyperparameter Exploration ---

--- Tuning Optimization Complete! Best Hyperparameters: ---
-> GRU Layer 1 Units: 96
-> Use LayerNorm 1  : False
-> GRU Layer 2 Units: 16
-> Dense Layer Units: 32
-> Learning Rate    : 0.01

--- Training Best Configured Model for Full Epochs ---


d:\New folder\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/300
41/41 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - loss: 0.1049 - mae: 0.2508 - val_loss: 0.0455 - val_mae: 0.1691 - learning_rate: 0.0100
Epoch 2/300
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.0536 - mae: 0.1956 - val_loss: 0.0229 - val_mae: 0.1318 - learning_rate: 0.0100
Epoch 3/300
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0503 - mae: 0.1920 - val_loss: 0.0226 - val_mae: 0.1309 - learning_rate: 0.0100
Epoch 4/300
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0473 - mae: 0.1859 - val_loss: 0.0199 - val_mae: 0.1223 - learning_rate: 0.0100
Epoch 5/300
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 0.0395 - mae: 0.1683 - val_loss: 0.0110 - val_mae: 0.0835 - learning_rate: 0.0100
Epoch 6/300
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0225 - mae: 0.1223 - val_loss: 0.0048 - val_mae: 0.0521 - learning_rate: 0.0100
Epoch 7/300
41/41 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0139 - mae: 0.0953 - val_loss: 0.0059 - val_mae: 0.0558 - learning_rate: 0.0100
Epoch 